In [1]:
import os
import requests
import pandas as pd
from datetime import datetime
from pytz import timezone
from dotenv import load_dotenv
from sqlalchemy import create_engine

In [ ]:
# Test Open-Meteo API with sample coordinates to check connection and JSON structure.


# Test coordinates
latitude = 41.01
longitude = 28.97

# Open-Meteo API URL
url = f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current_weather=true"

# Execute request
response = requests.get(url)

print("Status Code:", response.status_code)
print("Request Answer (JSON):", response.json())

Status Code: 200
Request Answer (JSON): {'latitude': 41.0, 'longitude': 29.0, 'generationtime_ms': 0.08416175842285156, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 48.0, 'current_weather_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature': '°C', 'windspeed': 'km/h', 'winddirection': '°', 'is_day': '', 'weathercode': 'wmo code'}, 'current_weather': {'time': '2026-09-28T17:15', 'interval': 900, 'temperature': 17.6, 'windspeed': 21.3, 'winddirection': 332, 'is_day': 0, 'weathercode': 61}}


In [ ]:
# Extract temperature, wind speed, and measurement time from the JSON response.

# Assign JSON answer to a variable
data = response.json()

# Pull current weather dictionary
current = data['current_weather']

# Store required metrics in variables
temperature = current['temperature']
windspeed = current['windspeed']
time = current['time']

print(f"Temperature: {temperature} °C")
print(f"Wind: {windspeed} km/h")
print(f"Time: {time}")

Temperature: 17.6 °C
Wind: 21.3 km/h
Time: 2026-09-28T17:15


In [ ]:
# Connect to Gans MySQL database and retrieve city names with their raw coordinates.

import os
import mysql.connector
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Connect to MySQL database
db_connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password=os.getenv("MYSQL_PASSWORD"),
    database="Gans"
)

cursor = db_connection.cursor()

# Query city information
cursor.execute("SELECT city_id, city_name, latitude, longitude FROM cities")
sehirler = cursor.fetchall()

# Display cities fetched from database
print("Cities from Gans Database:")
for sehir in sehirler:
    print(sehir)

Cities from Gans Database:
(1, 'Berlin', '52°31′12″N', '13°24′18″E')
(2, 'Hamburg', '53°33′N', '10°00′E')
(3, 'Munich', '48°08′15″N', '11°34′30″E')


In [ ]:
# Convert DMS coordinate strings into decimal float values for API queries.

import re

def dms_to_decimal(dms_str):
    # Use regex to match degrees, minutes, seconds, and direction indicator
    match = re.match(r"(\d+)°(?:(\d+)′)?(?:(\d+)″)?([NSEW])", str(dms_str))
    if not match:
        return float(dms_str) # Return as float directly if already numeric
    
    degrees, minutes, seconds, direction = match.groups()
    
    deg = float(degrees)
    min_val = float(minutes) if minutes else 0.0
    sec_val = float(seconds) if seconds else 0.0
    
    # Decimal calculation formula: Degree + (Minute / 60) + (Second / 3600)
    decimal = deg + (min_val / 60.0) + (sec_val / 3600.0)
    
    # Negate value for South (S) and West (W) coordinates
    if direction in ['S', 'W']:
        decimal *= -1
        
    return round(decimal, 4)

# Test coordinate conversion with cities retrieved from MySQL
for city_id, sehir, lat, lon in sehirler:
    lat_dec = dms_to_decimal(lat)
    lon_dec = dms_to_decimal(lon)
    print(f"[{city_id}] {sehir} -> Latitude: {lat_dec}, Longitude: {lon_dec}")

[1] Berlin -> Latitude: 52.52, Longitude: 13.405
[2] Hamburg -> Latitude: 53.55, Longitude: 10.0
[3] Munich -> Latitude: 48.1375, Longitude: 11.575


In [ ]:
# Loop through each city, convert coordinates, fetch current weather data 
# from Open-Meteo API andprint results.


# Loop through cities retrieved from Gans database
for city_id, sehir_adi, lat, lon in sehirler:
    # Convert DMS coordinates to decimal values
    lat_dec = dms_to_decimal(lat)
    lon_dec = dms_to_decimal(lon)
    
    # Construct Open-Meteo API request URL
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat_dec}&longitude={lon_dec}&current_weather=true"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        current = data['current_weather']
        
        temp = current['temperature']
        wind = current['windspeed']
        time = current['time']
        
        print(f"[{sehir_adi}] Temperature: {temp}°C | Wind: {wind} km/h | Time: {time}")
    else:
        print(f"[{sehir_adi}] Failed to fetch data! Status Code: {response.status_code}")

[Berlin] Temperature: 22.5°C | Wind: 7.0 km/h | Time: 2026-09-28T17:15
[Hamburg] Temperature: 21.8°C | Wind: 5.4 km/h | Time: 2026-09-28T17:15
[Munich] Temperature: 22.8°C | Wind: 9.8 km/h | Time: 2026-09-28T17:15


In [ ]:
# Weather Table ahs been created on the sql side

In [15]:
import requests

# SQL insert query template
insert_query = """
INSERT INTO weather (city_id, temperature, windspeed, recorded_at)
VALUES (%s, %s, %s, %s)
"""

# Fetch cities again to ensure cursor is fresh and loop through them
cursor.execute("SELECT city_id, city_name, latitude, longitude FROM cities")
sehirler = cursor.fetchall()

for city_id, city_name, lat, lon in sehirler:
    # Convert DMS coordinates to decimal values
    lat_dec = dms_to_decimal(lat)
    lon_dec = dms_to_decimal(lon)
    
    # Construct Open-Meteo API URL
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat_dec}&longitude={lon_dec}&current_weather=true"
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        current = data['current_weather']
        
        temp = current['temperature']
        wind = current['windspeed']
        time = current['time']
        
        # Execute insert query with city metrics
        cursor.execute(insert_query, (city_id, temp, wind, time))
        print(f"[{city_name}] Temperature: {temp}°C, Wind: {wind} km/h added to database.")
    else:
        print(f"[{city_name}] API failed!")

# Commit changes to persist inserted records in MySQL database
db_connection.commit()
print("\nAll data added successfully!")

[Berlin] Temperature: 22.1°C, Wind: 7.0 km/h added to database.
[Hamburg] Temperature: 21.6°C, Wind: 4.7 km/h added to database.
[Munich] Temperature: 22.1°C, Wind: 9.4 km/h added to database.

All data added successfully!


In [16]:
# PURPOSE: Query and display recorded weather data combined with city names using an SQL JOIN query.

# SQL query joining weather and cities tables
select_query = """
SELECT c.city_name, w.temperature, w.windspeed, w.recorded_at, w.created_at 
FROM weather w
JOIN cities c ON w.city_id = c.city_id;
"""

cursor.execute(select_query)
kayitlar = cursor.fetchall()

print("--- Gans Database Weather Records ---")
for kayit in kayitlar:
    print(f"City: {kayit[0]} | Temp: {kayit[1]}°C | Wind: {kayit[2]} km/h | Recorded At: {kayit[3]} | Created At: {kayit[4]}")

--- Gans Database Weather Records ---
City: Berlin | Temp: 22.1°C | Wind: 7.0 km/h | Recorded At: 2026-09-28T17:30 | Created At: 2026-09-28 19:36:46
City: Hamburg | Temp: 21.6°C | Wind: 4.7 km/h | Recorded At: 2026-09-28T17:30 | Created At: 2026-09-28 19:36:46
City: Munich | Temp: 22.1°C | Wind: 9.4 km/h | Recorded At: 2026-09-28T17:30 | Created At: 2026-09-28 19:36:46


In [ ]:
# Now lets take the 5 day forecast for the cities we work on

In [17]:
import requests

# Test coordinates (Berlin)
latitude = 52.52
longitude = 13.405

# Open-Meteo API URL configured for a 5-day daily forecast
url = f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&daily=temperature_2m_max,temperature_2m_min,windspeed_10m_max&forecast_days=5&timezone=auto"

response = requests.get(url)

print("Status Code:", response.status_code)
if response.status_code == 200:
    data = response.json()
    daily_data = data['daily']
    
    # Display forecast data line by line
    for i in range(len(daily_data['time'])):
        date = daily_data['time'][i]
        temp_max = daily_data['temperature_2m_max'][i]
        temp_min = daily_data['temperature_2m_min'][i]
        wind_max = daily_data['windspeed_10m_max'][i]
        print(f"Date: {date} | Max Temp: {temp_max}°C | Min Temp: {temp_min}°C | Max Wind: {wind_max} km/h")

Status Code: 200
Date: 2026-09-28 | Max Temp: 24.8°C | Min Temp: 12.5°C | Max Wind: 12.0 km/h
Date: 2026-09-29 | Max Temp: 25.1°C | Min Temp: 13.6°C | Max Wind: 14.8 km/h
Date: 2026-09-30 | Max Temp: 23.3°C | Min Temp: 13.6°C | Max Wind: 17.3 km/h
Date: 2026-10-01 | Max Temp: 22.1°C | Min Temp: 12.5°C | Max Wind: 9.9 km/h
Date: 2026-10-02 | Max Temp: 20.8°C | Min Temp: 15.2°C | Max Wind: 7.7 km/h


In [ ]:
Force close open MySQL connections and cursors to release table locks.

try:
    cursor.close()
    db_connection.close()
    print("Database connection successfully closed and released!")
except Exception as e:
    print("Connection was already closed or not found:", e)

Database connection successfully closed and released!


In [ ]:
# Retrieve city coordinates, fetch 5-day daily weather forecasts from Open-Meteo
# store them in the weather table.

import os
import requests
import mysql.connector
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Connect to Gans database
db_connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password=os.getenv("MYSQL_PASSWORD"),
    database="Gans"
)

cursor = db_connection.cursor()

# Query cities from database
cursor.execute("SELECT city_id, city_name, latitude, longitude FROM cities")
sehirler = cursor.fetchall()

# SQL query template for 5-day forecast insertion
insert_query = """
INSERT INTO weather (city_id, forecast_date, temp_max, temp_min, windspeed_max)
VALUES (%s, %s, %s, %s, %s)
"""

for city_id, city_name, lat, lon in sehirler:
    # Convert DMS coordinates to decimal values
    lat_dec = dms_to_decimal(lat)
    lon_dec = dms_to_decimal(lon)
    
    # API URL for 5-day forecast
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat_dec}&longitude={lon_dec}&daily=temperature_2m_max,temperature_2m_min,windspeed_10m_max&forecast_days=5&timezone=auto"
    
    response = requests.get(url)
    
    if response.status_code == 200:
        data = response.json()
        daily_data = data['daily']
        
        # Loop through the 5 forecast days
        for i in range(len(daily_data['time'])):
            date = daily_data['time'][i]
            temp_max = daily_data['temperature_2m_max'][i]
            temp_min = daily_data['temperature_2m_min'][i]
            wind_max = daily_data['windspeed_10m_max'][i]
            
            # Insert record into weather table
            cursor.execute(insert_query, (city_id, date, temp_max, temp_min, wind_max))
        
        print(f"[{city_name}] 5-day forecast added successfully.")
    else:
        print(f"[{city_name}] API request failed with status: {response.status_code}")

# Commit changes to database
db_connection.commit()
print("\nAll 5-day weather forecasts saved successfully!")

[Berlin] 5-day forecast added successfully.
[Hamburg] 5-day forecast added successfully.
[Munich] 5-day forecast added successfully.

All 5-day weather forecasts saved successfully!


In [ ]:
cursor.close()
db_connection.close()
print("Database connection successfully closed.")

Database connection successfully closed.
